## What this notebook does

**GOAL: Match users and their data contribution to demographics of their home location, leading to median datapoint contributions and demographics per census tract.**  

**WHEN: Use this notebook once per _region of interest (roi)_ and _period of interest (poi)_.**  

### Inputs

* `dpoints_homelocs/roi_yyyy_mm.csv`, from notebook `couple_users_homelocs.ipynb`
- Census gov ACS: <a href="https://data.census.gov/table?q=S1701+2019+ACS&g=050XX00US36005,36061,36081,36085,36047">S1701</a> | __Poverty table__  
  \> `Main/census-data/poverty-data/`*roi*`/ACSST5Y`*yyyy*`.S1701-Data_CensusTracts`*roi*`.csv` 
* Census gov ACS: <a href='https://data.census.gov/table?q=2021+S0101&g=0100000US$1400000'>S0101</a> | __Age and Sex table__   
  \> `Main/census-data/age-sex-data/ACSST5Y`*yyyy*`.S0101-Data_CensusTracts`*roi*`.csv` 

- Census gov ACS: <a href='https://data.census.gov/table?q=2021+S1501&g=0100000US$1400000'>S1501</a> | __Educational Attainment and Ethnicity table__  
  \> `Main/census-data/education-data/ACSST5Y`*yyyy*`.S1501-Data_CensusTracts`*roi*`.csv` 


### Outputs

- `stats/demographics/roi_yyyy_mm.csv`, roi level; overview of census tracts' total and median datapoints + demographics (poverty, age, sex, educational attainment, ethnicity).

## Settings

In [ ]:
path = '/home/jovyan/Main/mobility-bias_K'

In [ ]:
# minimum population size census tract
min_pop = 500

In [ ]:
# region of interest 
# roi = 'NYC'
# roi = 'LA'
roi = 'Chicago'
# roi = 'Houston'
# roi = 'Phoenix'
# roi = 'Philadelphia'
# roi = 'San_Antonio'
# roi = 'San_Diego'
# roi = 'Dallas'
# roi = 'San_Jose'

In [ ]:
# period of interest
# April 2019 - pre COVID
period = '2019_04'
# period = '2019_10'

# April 2020 - during COVID lockdown/start lockdown
# period = '2020_04'
# period = '2020_10'

# April 2021 - late COVID
# period = '2021_04'
# period = '2021_10'

# April 2022 - post COVID
# period = '2022_04'
# period = '2022_10'

In [ ]:
file_name_US = 'US_' + period
file_name_roi = roi + '_' + period

file_name_export = file_name_roi

## Setup

In [ ]:
import os
os.chdir(path)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
demo_year = period[:4]

## Mapping for external data census tract codes

In [ ]:
# Necessary for age_sex and edu_eth datafile conversion
# first create mapping from US state ids (FIPS state codes) to states admin1_names.
id2name_map = dict([('01', 'AL'), ('02', 'AK'), ('04', 'AZ'), ('05', 'AR'), ('06', 'CA'), ('08', 'CO'), ('09', 'CT'), ('10', 'DE'), ('11', 'DC'),
                    ('12', 'FL'), ('13', 'GA'), ('15', 'HI'), ('16', 'ID'), ('17', 'IL'), ('18', 'IN'), ('19', 'IA'), ('20', 'KS'), ('21', 'KY'),
                    ('22', 'LA'), ('23', 'ME'), ('24', 'MD'), ('25', 'MA'), ('26', 'MI'), ('27', 'MN'), ('28', 'MS'), ('29', 'MO'), ('30', 'MT'),
                    ('31', 'NE'), ('32', 'NV'), ('33', 'NH'), ('34', 'NJ'), ('35', 'NM'), ('36', 'NY'), ('37', 'NC'), ('38', 'ND'), ('39', 'OH'),
                    ('40', 'OK'), ('41', 'OR'), ('42', 'PA'), ('44', 'RI'), ('45', 'SC'), ('46', 'SD'), ('47', 'TN'), ('48', 'TX'), ('49', 'UT'),
                    ('50', 'VT'), ('51', 'VA'), ('53', 'WA'), ('54', 'WV'), ('55', 'WI'), ('56', 'WY')])

# define function to handle GEO_ids to sync with IDs used by data_provider
def admin_mapper(str_):
    # function to map id from US census poverty data, 
    # e.g. 1400000US01001020100 to census id in mobility data, e.g. US.AL.157.020700.1
    # get state mapping from iso2 number to state name2
    
    state_name = id2name_map.get(str_[9:11],'NaN')
    
    if state_name != 'NaN': # if state name is not nan
        return state_name + str_[11:] 
    else:
        return np.nan

## Age and sex datafile
ACS <a href="https://data.census.gov/table?q=S1701+2019+ACS&g=050XX00US36005,36061,36081,36085,36047">S1701</a>: Poverty table  
Column name codes:

In [ ]:
# load census data 
pov = pd.read_csv(f'../census-data/poverty-data/{roi}/ACSST5Y{period[0:4]}.S1701-Data_CensusTracts{roi}.csv', skiprows=[1])
    
# handle missing values 
print(f'Census tracts poverty table total: {len(pov.GEO_ID.unique())}')
pov_new = pov[pov['S1701_C03_001E'] != '-'][['GEO_ID','S1701_C03_001E']].reset_index(drop=True)
print(f'Census tracts with poverty data:   {len(pov_new.GEO_ID.unique())}')
pov = pov_new
del pov_new

# poverty percentages; rename column and transform to floats
pov.rename(columns = {'S1701_C03_001E':'poverty'}, inplace = True)
pov['poverty'] = pov['poverty'].astype(float)

# transform GEO_IDs to match how data_provider is storing census ids
pov['GEO_ID'] = pov['GEO_ID'].map(lambda x: admin_mapper(x))

display(pov) # pov on census_tract level

## Age and sex datafile
ACS <a href='https://data.census.gov/table?q=2021+S0101&g=0100000US$1400000'>S0101</a>: Age and Sex table  
Column name codes:

In [ ]:
# final column names to obtain
age_groups = ['<18', '18-29', '30-49', '50-64', '65+']
colnames = ['GEO_ID', 'poptot', 'female'] + age_groups

In [ ]:
# import age and sex data
# ACSST5Y2021.S0101-Data_CensusTractsNYC.csv
age_sex = pd.read_csv(f'../census-data/age-sex-data/ACSST5Y{demo_year}.S0101-Data_CensusTracts{roi}.csv', 
                      skiprows=[1], dtype=str)

# select necessary columns
# selecting: GEO_ID, total population all ages, female, 
#            <18, 18 - 24, 25 - 29, 30 - 34, 35 - 39, 40 - 44, 
#            45 - 49, 50 - 54, 55 - 59, 60 - 64, 65 +
age_sex = age_sex[['GEO_ID', 'S0101_C01_001E', 'S0101_C05_001E', 
                   'S0101_C01_022E', 'S0101_C01_023E', 'S0101_C01_007E', 'S0101_C01_008E', 'S0101_C01_009E', 'S0101_C01_010E', 
                   'S0101_C01_011E', 'S0101_C01_012E', 'S0101_C01_013E', 'S0101_C01_014E', 'S0101_C01_030E']]

# convert strings to integers
age_sex.iloc[:, 1:] = age_sex.iloc[:, 1:].astype(int)

In [ ]:
# # OPTIONAL plot population sizes distribution
# import matplotlib.pyplot as plt

# plt.hist(age_sex['S0101_C01_001E'], bins=np.linspace(0,18000,100))
# plt.axvline(min_pop, color='red', ls='--', lw=1)
# plt.xlabel('population size')
# plt.show()
# print(f'cut-off minimal pop = {min_pop}')

In [ ]:
# remove census tracts with extremely low populations
age_sex_new = age_sex[age_sex['S0101_C01_001E'] >= min_pop].reset_index(drop=True)

print(f"removed; pop < {min_pop}: {len(age_sex) - len(age_sex_new)} census tracts.")
age_sex = age_sex_new
del age_sex_new

In [ ]:
age_sex

In [ ]:
collapse_age = {'< 18': ['S0101_C01_022E'],
                '18-29': ['S0101_C01_023E', 'S0101_C01_007E'],
                '30-49': ['S0101_C01_008E', 'S0101_C01_009E', 'S0101_C01_010E', 'S0101_C01_011E'],
                '50-64': ['S0101_C01_012E', 'S0101_C01_013E', 'S0101_C01_014E'],
                '65+': ['S0101_C01_030E']}

In [ ]:
# merge columns of 5yr age categories to 5 age groups
for key, value in collapse_age.items():
    age_sex[value[0]] = age_sex[value].sum(axis=1)
    age_sex = age_sex.drop(value[1:], axis=1)

# give columns appropriate names
age_sex.columns = colnames
del collapse_age

# absolute values to percentages (relative to pop_tot, then delete pop_tot)
percentcols = colnames[2:]
age_sex[percentcols] = age_sex[percentcols].divide(age_sex.poptot, axis=0) * 100
# age_sex = age_sex.drop('poptot', axis=1)
del colnames

# transform GEO_IDs to match how data_provider is storing census ids
age_sex['GEO_ID'] = age_sex['GEO_ID'].map(lambda x: admin_mapper(x))

In [ ]:
age_sex

## Education and ethnicity datafile
ACS <a href='https://data.census.gov/table?q=2021+S1501&g=0100000US$1400000&tid=ACSST5Y2021.S1501'>S1501</a>: Educational Attainment table  
Column name codes:

In [ ]:
# final column names to obtain
colnames = ['GEO_ID', 'academic', 'white', 'black', 'asian', 'other']

In [ ]:
# import education and ethnicity
# ACSST5Y2021.S1501-Data_CensusTractsNYC.csv
edu_eth = pd.read_csv(f'../census-data/education-data/ACSST5Y{demo_year}.S1501-Data_CensusTracts{roi}.csv', 
                      skiprows=[1], dtype=str)

# selecting: GEO_ID, total population 25+, 25+ academic, 
#            white, black, american indian or alaska native, asian, 
#            native hawaiian and other pacific islander, some other race, multiracial
edu_eth = edu_eth[['GEO_ID', 'S1501_C01_006E', 'S1501_C01_015E',                           # education 
                   'S1501_C01_028E', 'S1501_C01_034E', 'S1501_C01_037E', 'S1501_C01_040E', # ethnicity
                   'S1501_C01_043E', 'S1501_C01_046E', 'S1501_C01_049E']]                  # ethnicity

# convert strings to integers
edu_eth.iloc[:, 1:] = edu_eth.iloc[:, 1:].astype(int)

# remove census tracts with 0 25+ population # OR MAKE < MIN_POP??
edu_eth = edu_eth[edu_eth['S1501_C01_006E'] > 0].reset_index(drop=True)

In [ ]:
edu_eth

In [ ]:
# calculate right numbers for custom ethnicity categories
edu_eth.iloc[:,8] = edu_eth.iloc[:,8] + edu_eth.iloc[:,5] + edu_eth.iloc[:,7] + edu_eth.iloc[:,9]
edu_eth = edu_eth.drop(columns=edu_eth.columns[[5, 7, 9]])

# percentages
percentcols = edu_eth.columns[1:]
edu_eth[percentcols] = edu_eth[percentcols].divide(edu_eth.iloc[:,1], axis=0) * 100
edu_eth = edu_eth.drop(columns=edu_eth.columns[1])
del percentcols

# give column names
edu_eth.columns = colnames
del colnames

# transform GEO_IDs to match how data_provider is storing census ids
edu_eth['GEO_ID'] = edu_eth['GEO_ID'].map(lambda x: admin_mapper(x))
edu_eth = edu_eth.dropna()

In [ ]:
edu_eth

## Datapoints and home location datafiles

In [ ]:
# import datapoints per user and their coupled poverty percentages
dpoints_homelocs = pd.read_csv(f'dpoints_homelocs/{file_name_roi}.csv')
dpoints_homelocs['data_points_tot'] = dpoints_homelocs['data_points']

# aggregate datapoints per census tract
dpoints = dpoints_homelocs.groupby('home_census_tract') \
                            .agg({'data_points_tot': 'sum',
                                  'data_points':'median'}) \
                            .reset_index().rename(columns={'home_census_tract': 'census_tract'})

dpoints

In [ ]:
# # OPTIONAL plot distribution median datapoints of census tracts, including mean and median
# import matplotlib.pyplot as plt

# plt.hist(user_pov.data_points, bins=np.arange(0,10000,10))
# plt.axvline(user_pov.data_points.median(), color='red')
# plt.axvline(user_pov.data_points.mean(), color='blue')
# plt.xlim(None, 10000)

## Combine data

In [ ]:
# combine all data
data = dpoints.set_index('census_tract').join(pov.set_index('GEO_ID').join(age_sex.set_index('GEO_ID').join(edu_eth.set_index('GEO_ID'))))
data = data.dropna()
data.reset_index(drop=False, inplace=True)

display(data)

In [ ]:
del dpoints, pov, age_sex, edu_eth, age_groups

In [ ]:
# put in whished order
columnorder = ['census_tract','poptot','data_points_tot','data_points',
               'female','poverty',
               '<18', '18-29', '30-49', '50-64', '65+',
               'academic',
               'white','black','asian','other']

data = data[columnorder]
data

In [ ]:
# save data
data.to_csv(f'stats/demographics/{file_name_export}.csv', index=False)

In [ ]:
# check how many users (and census tracts) lost because of missing census gov demographics data
users_before = len(dpoints_homelocs)
users_after = len(dpoints_homelocs[dpoints_homelocs.home_census_tract.isin(data.census_tract)])

print(f'Total users before adding demographics data: {users_before}')
print(f'Total users after:                           {users_after}')

users_rem = users_before - users_after
rem_rate = users_rem / users_before * 100
print(f'User removal rate: {rem_rate:.2f}%\n')

# possible: check how many census tracts removed
trcts_before = len(dpoints_homelocs.home_census_tract.unique())
trcts_after = len(data.census_tract.unique())
trcts_rem = trcts_before - trcts_after

print(f'Total tracts before adding demographics data: {trcts_before}')
print(f'Total tracts after:                           {trcts_after}')
print(f'Removed census tracts: {trcts_rem}')

In [ ]:
# save removal numbers
# of removing census tracts without demographic information AND census tracts with populations < 500
file = f'stats/2019_04_removal_rates_nodemos.txt'

newlines = [
    f'{file_name_export}',
    f' users_after_out={users_before} trcts_after_out={trcts_before} rem_nodemo_trcts={trcts_rem}',
    f' rem_nodemo={users_rem} rem_nodemo_rate={rem_rate:.3f}%'
]

try: 
    with open(file) as f:
        data = f.read().splitlines()
except:
    data = []

if file_name_export in data:
    for i, newline in enumerate(newlines):
        data[data.index(file_name_export) + i] = newline
else:
    [data.insert(i, newline) for i, newline in enumerate(newlines)]

with open(file, 'w') as f:
    for l in data:
        f.write(l + '\n')

In [ ]:
# calculate total removal rates
start_file = f'stats/{period}_removal_rates_outliers.txt'
with open(start_file) as f:
    start_data = f.read().splitlines()

users_start = int(start_data[start_data.index(file_name_export) + 1].split(' ')[1][12:])
final_rem_rate = (users_start - users_after) / users_start * 100

print(f'Final user number: {users_after} of initial {users_start}')
print(f'Total removal rate: {final_rem_rate:.2f}%')

In [ ]:
# save total removal rates
file = f'stats/{period}_removal_rates_final.txt'

newlines = [
    f'{file_name_export}',
    f' final_users={users_after} final_rem_rate={final_rem_rate:.3f}% final_trcts={trcts_after}'
]

try: 
    with open(file) as f:
        data = f.read().splitlines()
except:
    data = []

if file_name_export in data:
    for i, newline in enumerate(newlines):
        data[data.index(file_name_export) + i] = newline
else:
    [data.insert(i, newline) for i, newline in enumerate(newlines)]

with open(file, 'w') as f:
    for l in data:
        f.write(l + '\n')